In [ ]:
import spacy
import stanza
from collections import Counter, defaultdict
import csv
import re
import os
import pandas as pd
import numpy as np
import random

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])


/home/p318482/syntactic-bootstrapping/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Regex for tokenizing words
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

In [5]:
def load_verb_bins(binned_data_dir):
    """
    Load verb bins from a nested folder structure.

    Directory structure expected:
    binned_data_dir/
        POS/
            TAG/
                bin_0.csv
                bin_1.csv
                ...

    Returns:
        dict: {(pos, tag, bin_idx): [verbs]}
    """
    verb_bins = defaultdict(list)

    for pos_folder in os.listdir(binned_data_dir):
        pos_path = os.path.join(binned_data_dir, pos_folder)
        if not os.path.isdir(pos_path):
            continue
        for tag_folder in os.listdir(pos_path):
            tag_path = os.path.join(pos_path, tag_folder)
            if not os.path.isdir(tag_path):
                continue
            for bin_file in os.listdir(tag_path):
                if not bin_file.endswith(".csv"):
                    continue
                try:
                    bin_idx = int(bin_file.split("_")[-1].split(".")[0])
                except ValueError:
                    print(f"⚠️ Skipping file (bad name): {bin_file}")
                    continue
                df = pd.read_csv(os.path.join(tag_path, bin_file))
                verbs = df["word"].dropna().astype(str).tolist()
                verb_bins[(pos_folder, tag_folder, bin_idx)].extend(verbs)

    print(f"Loaded bins for {len(verb_bins)} POS-TAG-bin combinations.")
    return verb_bins

In [ ]:
verbs_dist = load_verb_bins("./binned_data/CHILDES_rand")

verbs_dist['VERB', 'VBD', 5][:10]  # Example access

Loaded bins for 467 POS-TAG-bin combinations.


['fixed',
 'moved',
 'let',
 'pulled',
 'read',
 'spilled',
 'drew',
 'bumped',
 'slept',
 'washed']

In [ ]:
def get_verb_bin_for_verb(main_verb, verb_bins):
    """
    Return the bin key containing the original verb.
    If multiple bins contain it, pick one randomly.
    """
    pos, tag = main_verb.pos_, main_verb.tag_
    candidate_bins = [
        key for key in verb_bins.keys()
        if key[0] == pos and key[1] == tag and main_verb.text.lower() in [v.lower() for v in verb_bins[key]]
    ]
    if not candidate_bins:
        return None
    return random.choice(candidate_bins)

def generate_minimal_pairs(
    dataset_path: str,
    output_txt: str,
    verb_bins: dict,
    pattern: re.Pattern,
    spacy_model: str = "en_core_web_sm",
    min_tokens: int = 10,
    max_tokens: int = 30,
    num_replacements: int = 5
):
    """
    Generate minimal pairs for sentences in dataset_path.
    
    Each line in the output file will have: original_sentence,new_sentence

    Parameters:
    - dataset_path: path to input text file (one sentence per line)
    - output_txt: path to output TXT file
    - verb_bins: dict {(pos, tag, bin_idx): [verbs]} for sampling replacements
    - pattern: compiled regex pattern for tokenizing words
    - spacy_model: SpaCy model to use
    - min_tokens: minimum number of tokens to consider a sentence
    - num_replacements: number of minimal pair variations per sentence
    """
    
    punct_fix = re.compile(r"\s+([.,?!])")  # fix spacing before punctuation
    nlp = spacy.load(spacy_model, disable=["ner"])


    with open(dataset_path, "r", encoding="utf-8") as f, open(output_txt, "w", encoding="utf-8", newline="") as out_f:
        writer = csv.writer(out_f, delimiter=",")
        
        for line in f:
            line = line.strip()
            if not line:
                continue

            tokens = pattern.findall(line)
            if len(tokens) < min_tokens or len(tokens) > max_tokens:
                continue  # skip short sentences

            doc = nlp(line)

            # Identify main verb (ROOT)
            main_verb = None
            for token in doc:
                if token.pos_ == "VERB" and token.dep_ == "ROOT":
                    main_verb = token
                    break
            if main_verb is None:
                continue

    
            bin_key = get_verb_bin_for_verb(main_verb, verb_bins)
            if bin_key is None:
                continue

            # Sample replacements from the same bin (exclude original)
            candidates = [v for v in verb_bins[bin_key] if v.lower() != main_verb.text.lower()]
            if not candidates:
                continue
            replacements = random.sample(candidates, min(num_replacements, len(candidates)))

            # Replace verbs at token level
            token_texts = [t.text for t in doc]
            for rep in replacements:
                new_tokens = token_texts.copy()
                new_tokens[main_verb.i] = str(rep)
                new_sent = " ".join(new_tokens)
                new_sent = punct_fix.sub(r"\1", new_sent)  # fix spacing before punctuation
                writer.writerow([line, new_sent])

    print(f"Minimal pairs saved to {output_txt}")

In [ ]:
def get_verb_bin_for_verb_stanza(main_verb, verb_bins):
        """
        Return the bin key containing the original verb.
        If multiple bins contain it, pick one randomly.
        """
        pos, tag = main_verb.upos, main_verb.xpos
        candidate_bins = [
            key for key in verb_bins.keys()
            if key[0] == pos and key[1] == tag and main_verb.text.lower() in [v.lower() for v in verb_bins[key]]
        ]
        if not candidate_bins:
            return None
        return random.choice(candidate_bins)

def generate_minimal_pairs_stanza(
    dataset_path: str,
    output_txt: str,
    verb_bins: dict,
    pattern: re.Pattern,
    nlp_stanza,
    min_tokens: int = 10,
    num_replacements: int = 5
):
    """
    Generate minimal pairs using Stanza parser.
    
    Each line in output_txt will contain: original_sentence, modified_sentence
    with standardized punctuation (no spaces before punctuation).
    """
    punct_fix = re.compile(r"\s+([.',?!])")


    with open(dataset_path, "r", encoding="utf-8") as f, open(output_txt, "w", encoding="utf-8", newline="") as out_f:
        writer = csv.writer(out_f, delimiter=",")
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = pattern.findall(line)
            if len(tokens) < min_tokens:
                continue  # skip short sentences

            doc = nlp_stanza(line)

            # Flatten tokens
            all_words = [word for sent in doc.sentences for word in sent.words]

            # Identify main verb = root of dependency tree
            main_verb = None
            for word in all_words:
                if word.upos == "VERB" and word.deprel == 'root':  # ROOT in Stanza
                    main_verb = word
                    break
            if main_verb is None:
                continue

            # Get bin for this verb
            bin_key = get_verb_bin_for_verb_stanza(main_verb, verb_bins)
            if bin_key is None:
                continue
            if bin_key is None:
                continue

            # Sample replacement verbs
            candidates = [v for v in verb_bins[bin_key] if str(v).lower() != main_verb.text.lower()]
            if len(candidates) < num_replacements:
                replacements = candidates
            else:
                replacements = random.sample(candidates, num_replacements)

            # Replace in token list
            original_sentence = " ".join([w.text for w in all_words])
            # Apply punctuation fix to original
            original_sentence = punct_fix.sub(r"\1", original_sentence)

            for rep in replacements:
                new_tokens = []
                for word in all_words:
                    if word.id == main_verb.id:  # replace only the root verb
                        new_tokens.append(str(rep))
                    else:
                        new_tokens.append(word.text)
                # Apply punctuation fix to modified sentence
                new_sentence = " ".join(new_tokens)
                new_sentence = punct_fix.sub(r"\1", new_sentence)

                writer.writerow([original_sentence, new_sentence])

    print(f"Minimal pairs saved to {output_txt}")

## CHILDES stanza
We have trained a in-domain Stanza Dependency Parser on official UD annotations for *English_CHILDES*. In Huggingface you can find it and download it; it is ready to be used with the stanza pipeline -> *fpadovani/cds_parser_roberta_stanza*

We extensively worked on releasing a Parser trained on golden English_CHILDES UD annotations and wrote a paper that is currently under review. But the repository of this work is under construction and you can have a look at this [link](https://github.com/fpadovani/CAIT-Toolkit). 

Also the POS model can be found in the repository under the *saved_model* folder.

In [ ]:
pos_tagger_model = './saved_models/pos/en_childes_charlm_tagger.pt'

# Dependency parse pipeline with your custom parser model
parser_model = './saved_models/depparse/en_childes_charlm_parser.pt'
nlp_childes = stanza.Pipeline(
    lang='en',
    processors='tokenize,pos,lemma,depparse',
    use_gpu=True,  # Set to True if GPU is available
    pos_model_path=pos_tagger_model,
    depparse_model_path=parser_model
)

2025-12-04 10:00:40 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-12-04 10:00:40 INFO: Downloaded file to /home/p318482/stanza_resources/resources.json
2025-12-04 10:00:40 WARNING: Language en package default expects mwt, which has been added
2025-12-04 10:00:41 INFO: Loading these models for language: en (English):
| Processor | Package                 |
---------------------------------------
| tokenize  | combined                |
| mwt       | combined                |
| pos       | /home/p318..._tagger.pt |
| lemma     | combined_nocharlm       |
| depparse  | /home/p318..._parser.pt |

2025-12-04 10:00:41 INFO: Using device: cuda
2025-12-04 10:00:41 INFO: Loading: tokenize
2025-12-04 10:00:44 INFO: Loading: mwt
2025-12-04 10:00:44 INFO: Loading: pos
2025-12-04 10:00:48 INFO: Loading: lemma
2025-12-04 10:00:49 INFO: Loading: deppa

## CHILDES

In [ ]:
binned_data_dir = "./binned_data/CHILDES_rand"
verb_bins = load_verb_bins(binned_data_dir)


dataset_path = "./corpora/CHILDES_rand/original/childes_rand.test.txt"
output_txt = "./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand/minimal_pairs_childes_rand.txt"

pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

generate_minimal_pairs_stanza(dataset_path, output_txt, verb_bins, pattern, nlp_childes)

## WIKI

In [ ]:
binned_data_dir = './binned_data/WIKI_rand'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./corpora/WIKI_rand/original/wiki_rand.test.txt"
output_txt = "./evaluation/semantic_minimal_pairs/data/verb_focus/WIKI_rand/minimal_pairs_wiki_rand.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)

## BNC

In [ ]:
binned_data_dir = './binned_data/BNC_rand'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./corpora/BNC_rand/original/bnc_rand.test.txt"
output_txt = "./evaluation/semantic_minimal_pairs/data/verb_focus/minimal_pairs_bnc_rand.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)

## CANDOR

In [ ]:
binned_data_dir = './binned_data/CANDOR_rand'
verb_bins = load_verb_bins(binned_data_dir)

dataset_path = "./corpora/CANDOR_rand/original/candor_rand.test.txt"
output_txt = "./evaluation/semantic_minimal_pairs/data/verb_focus/CANDOR_rand/minimal_pairs_candor_new.txt"

generate_minimal_pairs(dataset_path, output_txt, verb_bins, pattern)

After the creation of the minimal pair dataset we have to go through other steps of cleaning and checks.

In [ ]:
auxiliaries = [
    "am", "are", "is", "was", "were",
    "have", "has", "had",
    "do", "does", "did",
    "can", "could",
    "will", "would",
    "shall", "should",
    "may", "might",
    "must"
]

INSIDE_BRACKETS = re.compile(r"([\(\[\{])\s*(.*?)\s*([\)\]\}])")
INSIDE_QUOTES = re.compile(r"([\"'‘“”])\s*(.*?)\s*([\"'“’”])")
REMOVE_SPACE_BEFORE_PUNCT = re.compile(r"\s+([,.:;!?])")          # space before punctuation
punct_fix = re.compile(r"\s+([,.'?!])$")      # space before . ? !
ellipsis_fix = re.compile(r"\s+(\.\.\.)$")  # space before ...
underscores_fix = re.compile(r"_([A-Za-z0-9]+)_")  # remove surrounding underscores
aux_pattern = re.compile(r"\b(" + "|".join(auxiliaries) + r")\s+n['’]t\b", flags=re.IGNORECASE)
generic_clitic_pattern = re.compile(
    r"\b([A-Za-z]+)\s+('re|'m|'s|'ve|'d|'ll|n['’]t)\b", flags=re.IGNORECASE
)
# common split-to-joined contractions
split_to_joined = {
    r"\bwan\s+na\b": "wanna",
    r"\bgon\s+na\b": "gonna",
    r"\bgot\s+ta\b": "gotta",
    r"\blem\s+me\b": "lemme",
    r"\bgim\s+me\b": "gimme",
    r"\dun\s+no\s+know\b": "dunno",
    r"\bought\s+to\b": "oughta",
    r"\bcan\s+not\b": "cannot",
}
split_patterns = [(re.compile(k, flags=re.IGNORECASE), v) for k, v in split_to_joined.items()]
hyphen_fix_pattern = re.compile(r"\b([A-Za-z0-9]+)\s*-\s*([A-Za-z0-9]+)\b")



def clean_sentence(sentence: str) -> str:
    """Apply all cleaning rules to a single sentence."""
    sentence = sentence.strip().lower()
    sentence = punct_fix.sub(r"\1", sentence)
    sentence = ellipsis_fix.sub(r"\1", sentence)
    sentence = underscores_fix.sub(r"\1", sentence)
    sentence = aux_pattern.sub(lambda m: m.group(1).lower() + "n't", sentence)
    sentence = generic_clitic_pattern.sub(lambda m: m.group(1) + m.group(2), sentence)
    for pattern, replacement in split_patterns:
        sentence = pattern.sub(replacement, sentence)
    sentence = hyphen_fix_pattern.sub(r"\1-\2", sentence)
    if sentence.startswith("' "):
        sentence = sentence.replace("' ", "'", 1)
    if sentence.startswith('" '):
        sentence = sentence.replace('" ', '"', 1)
    if sentence.endswith('  . "'):
        sentence = sentence.replace('  . "', '."', 1)
    if ' ...' in sentence: 
        sentence = sentence.replace(' ...', '...', 1)
    if ' . ..' in sentence:
        sentence = sentence.replace(' . ..', '...', 1)
    if ' . . .' in sentence:
        sentence = sentence.replace(' . . .', '...', 1)
    sentence = REMOVE_SPACE_BEFORE_PUNCT.sub(r"\1", sentence)
    sentence = INSIDE_BRACKETS.sub(r"\1\2\3", sentence)
    sentence = INSIDE_QUOTES.sub(r"\1\2\3", sentence)
    sentence = re.sub(r"  +", " ", sentence)
    return sentence

def clean_minimal_pairs_csv(datasets: dict, output_dir: str):
    """Apply cleaning to both columns of minimal pairs CSV files."""
    os.makedirs(output_dir, exist_ok=True)

    for split, input_file in datasets.items():
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        output_file = os.path.join(output_dir, f"{base_name}.{split}.cleaned.csv")
        print(f"Processing {split} split for minimal pairs cleaning...")

        with open(input_file, "r", encoding="utf-8") as f_in, \
             open(output_file, "w", encoding="utf-8", newline="") as f_out:

            reader = csv.reader(f_in)
            writer = csv.writer(f_out)
            header = next(reader)  # keep header
            writer.writerow(header)

            for row in reader:
                if len(row) < 2:
                    continue
                s1, s2 = row[0], row[1]
                cleaned_s1 = clean_sentence(s1)
                cleaned_s2 = clean_sentence(s2)
                writer.writerow([cleaned_s1, cleaned_s2])

        print(f"Saved cleaned {split} to {output_file}")


In [ ]:
input_file = "./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand/minimal_pairs_childes_rand_new.txt"
output_file = "./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand/minimal_pairs_childes_rand_new_correct.txt"
max_lines = 140000
# Regex to match words (simple whitespace-based tokenization)
punct_fix = re.compile(r"\s+([.',?!])")

def count_tokens(sentence):
    # Remove leading/trailing whitespace
    sentence = sentence.strip()
    # Count tokens by splitting on whitespace
    tokens = sentence.split()
    # Optionally include punctuation fixes
    tokens += punct_fix.findall(sentence)
    return len(tokens)

with open(input_file, "r", encoding="utf-8") as f_in, \
     open(output_file, "w", newline="", encoding="utf-8") as f_out:

    reader = csv.DictReader(f_in, delimiter=',')
    fieldnames = reader.fieldnames
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()

    kept = 0
    skipped = 0

    for row in reader:
        if kept >= max_lines:
            break

        s1 = row["sentence1"]
        s2 = row["sentence2"]

        if count_tokens(s1) <= 30 and count_tokens(s2) <= 30:
            writer.writerow(row)
            kept += 1
        else:
            skipped += 1

print(f"Done! Kept {kept} rows, skipped {skipped} rows.")

Done! Kept 140000 rows, skipped 2291 rows.


In [ ]:
datasets = {
    "childes": "./evaluation/clm_semantic/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new_cleaned.txt"
}

output_dir = "./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand_new_new/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
datasets = {
    "candor": "./evaluation/semantic_minimal_pairs/data/verb_focus/BNC_SPOKEN_CANDOR/minimal_pairs_candor.txt"
}

output_dir = "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/BNC_SPOKEN_CANDOR/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
datasets = {
    "bnc": "./evaluation/semantic_minimal_pairs/data/verb_focus/BNC_rand/minimal_pairs_bnc_rand.txt"
}

output_dir = "./evaluation/semantic_minimal_pairs/data/verb_focus/BNC_rand/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
datasets = {
    "wiki": "./syntactic-bootstrapping/evaluation/clm_semantic/data/verb_focus/WIKI_rand/minimal_pairs_wiki_rand.txt"
}

output_dir = "./evaluation/semantic_minimal_pairs/data/verb_focus/WIKI_rand/cleaned"
clean_minimal_pairs_csv(datasets, output_dir)

In [ ]:
# Load your dataset
df = pd.read_csv("./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new.txt")  # or pd.read_excel(...)

# Use regex to filter out rows where 'sentence1' contains 'wan na'
# The regex r'\bwan\s+na\b' ensures we match 'wan na' as separate words
df_cleaned = df[~df['sentence1'].str.contains(r'\bwan\s+na\b', regex=True)]
df_cleaned = df[~df['sentence1'].str.contains(r'\bgot\s+ta\b', regex=True)]
df_cleaned = df[~df['sentence1'].str.contains(r'\bgon\s+na\b', regex=True)]

# Save the cleaned dataset
df_cleaned.to_csv("./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand_new_new/minimal_pairs_childes_rand_new_cleaned.txt", index=False)


In [ ]:
childes = "./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand/cleaned/minimal_pairs_childes_rand_cleaned.csv"
wiki = "./evaluation/semantic_minimal_pairs/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand_cleaned.csv"
bnc = "./evaluation/semantic_minimal_pairs/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand_cleaned.csv"
candor = "./evaluation/semantic_minimal_pairs/data/verb_focus/CANDOR_rand/cleaned/minimal_pairs_candor_rand_cleaned.csv"

In [ ]:
def normalize_sentence_for_comparison(sentence: str) -> str:
    """
    Lowercase and remove/normalize spaces and punctuation for token alignment.
    Keeps only word characters and single spaces.
    """
    sentence = sentence.lower().strip()
    # replace punctuation with spaces
    sentence = re.sub(r"[^\w\s']", " ", sentence)
    # collapse multiple spaces
    sentence = re.sub(r"\s+", " ", sentence)
    return sentence

def extract_target_verbs_with_pos_spacy_normalized(path, nlp, max_lines=140000):
    """
    Extract one target verb per unique sentence1 (minimal pair group),
    along with its lemma, coarse POS, and fine-grained POS using SpaCy.
    Normalizes sentences to avoid mismatches due to spaces/punctuation.
    """
    results = []
    seen_sentence1 = set()

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue

            s1, s2 = row[0].strip(), row[1].strip()

            if s1 in seen_sentence1:
                continue
            seen_sentence1.add(s1)

            # normalize sentences for token comparison
            s1_norm = normalize_sentence_for_comparison(s1)
            s2_norm = normalize_sentence_for_comparison(s2)

            # tokenize original sentences with SpaCy
            doc1 = nlp(s1_norm)
            doc2 = nlp(s2_norm)

            # flatten tokens
            tokens1 = [t for t in doc1]
            tokens2 = [t for t in doc2]

            # find first mismatch using normalized tokens
            for tok1, tok2 in zip(tokens1, tokens2):
                tok1_norm = normalize_sentence_for_comparison(tok1.text)
                tok2_norm = normalize_sentence_for_comparison(tok2.text)
                if tok1_norm != tok2_norm:
                    results.append({
                        'sentence1': s1,
                        'verb': tok1.text,
                        'lemma': tok1.lemma_,
                        'upos': tok1.pos_,
                        'xpos': tok1.tag_
                    })
                    break

    return results


In [ ]:
def extract_target_verbs_with_pos_spacy(path, nlp, max_lines=140000):
    """
    Extract one target verb per unique sentence1 (minimal pair group),
    along with its lemma, coarse POS, and fine-grained POS using SpaCy.
    
    Args:
        path: path to CSV file containing minimal pairs (sentence1, sentence2)
        nlp: SpaCy NLP pipeline with POS tagging
        max_lines: max number of lines to process
    Returns:
        List of dicts: [{'verb':..., 'lemma':..., 'upos':..., 'xpos':...}, ...]
    """
    results = []
    seen_sentence1 = set()

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue

            s1, s2 = row[0].strip(), row[1].strip()

            if s1 in seen_sentence1:
                continue
            seen_sentence1.add(s1)

            # tokenize with SpaCy
            doc1 = nlp(s1)
            doc2 = nlp(s2)

            # iterate over tokens and find first mismatch
            for tok1, tok2 in zip(doc1, doc2):
                if tok1.text != tok2.text:
                    results.append({
                        'sentence1': s1,
                        'verb': tok1.text,
                        'lemma': tok1.lemma_,
                        'upos': tok1.pos_,
                        'xpos': tok1.tag_
                    })
                    break

    return results

In [ ]:
def extract_target_verbs_with_pos_stanza(path, nlp, max_lines=140000):
    """
    Extract one target verb per unique sentence1 (minimal pair group), 
    along with its lemma, coarse POS (UPOS), and fine-grained POS (XPOS).
    
    Args:
        path: path to CSV file containing minimal pairs (sentence1, sentence2)
        nlp: stanza NLP pipeline with POS and depparse
        max_lines: max number of lines to process
    Returns:
        List of dicts: [{'verb':..., 'lemma':..., 'upos':..., 'xpos':...}, ...]
    """
    results = []
    seen_sentence1 = set()

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue

            s1, s2 = row[0].strip(), row[1].strip()

            if s1 in seen_sentence1:
                continue
            seen_sentence1.add(s1)


            # tokenize with stanza
            doc1 = nlp(s1)
            doc2 = nlp(s2)

            # flatten tokens (stanza outputs sentences)
            tokens1 = [t for sent in doc1.sentences for t in sent.tokens]
            tokens2 = [t for sent in doc2.sentences for t in sent.tokens]

            # iterate over tokens and find first mismatch
            for tok1, tok2 in zip(tokens1, tokens2):
                if tok1.text != tok2.text:
                    # take the first word of the token (usually just one)
                    word = tok1.words[0]
                    results.append({
                        'sentence1': s1,
                        'lemma': word.lemma,
                        'verb': word.text,
                        'upos': word.upos,
                        'xpos': word.xpos
                    })
                    break

    return results


In [ ]:
def load_binned_words(binned_dir):
    """
    Load all words from the binned CSVs into a nested dictionary:
    binned_words[coarse_pos][fine_tag][bin_idx] = list of words
    """
    binned_words = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    for pos in os.listdir(binned_dir):
        pos_path = os.path.join(binned_dir, pos)
        if not os.path.isdir(pos_path):
            continue
        for tag in os.listdir(pos_path):
            tag_path = os.path.join(pos_path, tag)
            if not os.path.isdir(tag_path):
                continue
            for bin_file in os.listdir(tag_path):
                if bin_file.endswith(".csv"):
                    bin_idx = int(bin_file.split("_")[1].split(".")[0])
                    df = pd.read_csv(os.path.join(tag_path, bin_file))
                    words = df["word"].tolist()
                    binned_words[pos][tag][bin_idx].extend(words)

    return binned_words




In [ ]:
def count_target_verbs_in_bins(target_verbs_info, binned_words):
    """
    Count how many target verbs fall into each bin (1-10).

    Args:
        target_verbs_info: list of dicts with keys: 'verb', 'lemma', 'upos', 'xpos'
        binned_words: nested dict binned_words[upos][xpos][bin_idx] = list of words

    Returns:
        dict {bin_1: count, bin_2: count, ..., bin_10: count}
    """
    bin_counts = defaultdict(int)

    for v in target_verbs_info:
        verb = v['verb']
        upos = v['upos']
        xpos = v['xpos']

        # check which bin the verb falls into
        found = False
        if upos in binned_words and xpos in binned_words[upos]:
            for bin_idx, words_in_bin in binned_words[upos][xpos].items():
                if verb in words_in_bin:
                    bin_counts[f'bin_{bin_idx}'] += 1
                    found = True
                    break  # assume each verb belongs to only one bin

        if not found:
            bin_counts['bin_not_found'] += 1  # optional: track verbs not in any bin

    # ensure all bins 1-10 exist (even if 0)
    for i in range(1, 11):
        bin_counts.setdefault(f'bin_{i}', 0)

    return dict(bin_counts)

In [ ]:
def count_target_verbs_in_bins_with_not_found(target_verbs_info, binned_words):
    """
    Count how many target verbs fall into each bin (1-10) and save the verbs
    that were not found in any bin.

    Args:
        target_verbs_info: list of dicts with keys: 'verb', 'lemma', 'upos', 'xpos'
        binned_words: nested dict binned_words[upos][xpos][bin_idx] = list of words

    Returns:
        bin_counts: dict {bin_1: count, ..., bin_10: count, bin_not_found: count}
        not_found_verbs: list of dicts [{'sentence1':..., 'verb':..., 'lemma':..., 'upos':..., 'xpos':...}, ...]
    """
    bin_counts = defaultdict(int)
    not_found_verbs = []

    for v in target_verbs_info:
        verb = v['verb']
        upos = v['upos']
        xpos = v['xpos']

        found = False
        if upos in binned_words and xpos in binned_words[upos]:
            for bin_idx, words_in_bin in binned_words[upos][xpos].items():
                if verb in words_in_bin:
                    bin_counts[f'bin_{bin_idx}'] += 1
                    found = True
                    break

        if not found:
            bin_counts['bin_not_found'] += 1
            not_found_verbs.append(v)

    # ensure all bins 1-10 exist
    for i in range(1, 11):
        bin_counts.setdefault(f'bin_{i}', 0)

    return dict(bin_counts), not_found_verbs


## CHILDES 

In [ ]:

bins_childes = load_binned_words('./binned_data/CHILDES_rand')
childes_verbs_info = extract_target_verbs_with_pos_stanza("./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand/cleaned/minimal_pairs_childes_rand_cleaned.csv", nlp_childes)
bin_counts_childes, not_found_verbs = count_target_verbs_in_bins_with_not_found(childes_verbs_info, bins_childes)
print("Bin counts:", bin_counts_childes)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("childes_bin_not_found.csv", index=False)

## WIKIPEDIA

In [ ]:
bins_wiki = load_binned_words('./syntactic-bootstrapping/binned_data/WIKI_rand')
wiki_verbs_info = extract_target_verbs_with_pos_spacy_normalized('./evaluation/semantic_minimal_pairs/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand_cleaned.csv', nlp)
bin_counts_wiki, not_found_verbs = count_target_verbs_in_bins_with_not_found(wiki_verbs_info, bins_wiki)
print("Bin counts:", bin_counts_wiki)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("wiki_bin_not_found.csv", index=False)

## BNC

In [ ]:
bins_bnc= load_binned_words('./syntactic-bootstrapping/binned_data/BNC_rand')
bnc_verbs_info = extract_target_verbs_with_pos_spacy_normalized('./evaluation/semantic_minimal_pairs/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand_cleaned.csv', nlp)
bin_counts_bnc, not_found_verbs = count_target_verbs_in_bins_with_not_found(bnc_verbs_info, bins_bnc)
print("Bin counts:", bin_counts_bnc)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("bnc_bin_not_found.csv", index=False)

## CANDOR

In [ ]:
bins_candor = load_binned_words('./syntactic-bootstrapping/corpora/CANDOR_rand')
candor_verbs_info = extract_target_verbs_with_pos_spacy_normalized('./evaluation/semantic_minimal_pairs/data/verb_focus/CANDOR_rand/cleaned/minimal_pairs_candor_rand_cleaned.csv', nlp, max_lines=140000)
bin_counts_candor, not_found_verbs = count_target_verbs_in_bins_with_not_found(candor_verbs_info, bins_candor)
print("Bin counts:", bin_counts_candor)
print("Number of verbs not found:", len(not_found_verbs))

pd.DataFrame(not_found_verbs).to_csv("candor_bin_not_found.csv", index=False)

## CREATE A NEW csv of minimal pairs where you control also for frequency bins of the verbs

In [ ]:
def create_minimal_pair_with_bin(input_csv, output_csv, target_verbs_info, binned_words, max_lines=140000):
    """
    Create a new minimal pair CSV with an extra column 'bin_number' indicating
    the bin of the target verb in sentence1.
    
    Args:
        input_csv: path to original minimal pairs CSV (sentence1, sentence2)
        output_csv: path to save new CSV (sentence1, sentence2, bin_number)
        target_verbs_info: list of dicts [{'sentence1':..., 'verb':..., 'lemma':..., 'upos':..., 'xpos':...}]
        binned_words: nested dict binned_words[upos][xpos][bin_idx] = list of words
        max_lines: max lines to process
    """

    # Build a mapping from sentence1 to its bin_number
    s1_to_bin = {}
    for info in target_verbs_info:
        s1 = info['sentence1']
        if s1 in s1_to_bin:
            continue  # already assigned
        verb = info['verb']
        upos = info['upos']
        xpos = info['xpos']
        bin_number = 'bin_not_found'
        if upos in binned_words and xpos in binned_words[upos]:
            for bin_idx, words_in_bin in binned_words[upos][xpos].items():
                if verb in words_in_bin:
                    bin_number = f'bin_{bin_idx}'
                    break
        s1_to_bin[s1] = bin_number

    # Read original CSV and write new CSV with bin_number
    with open(input_csv, "r", encoding="utf-8") as fin, \
         open(output_csv, "w", encoding="utf-8", newline='') as fout:
        reader = csv.reader(fin)
        writer = csv.writer(fout)
        header = next(reader)
        writer.writerow(header + ['bin_number'])

        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            if len(row) < 2:
                continue
            s1, s2 = row[0].strip(), row[1].strip()
            bin_number = s1_to_bin.get(s1, 'bin_not_found')
            writer.writerow([s1, s2, bin_number])

    print(f"Saved minimal pair CSV with bin numbers to: {output_csv}")


In [ ]:
create_minimal_pair_with_bin(
    input_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand_new_new/cleaned/childes_cleaned_2.csv",
    output_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/CHILDES_rand_new_new/cleaned/childes_cleaned_2_with_bins.csv",
    target_verbs_info=childes_verbs_info,
    binned_words=bins_childes,
    max_lines=140000
)

In [ ]:
create_minimal_pair_with_bin(
    input_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand_cleaned.csv",
    output_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/BNC_rand/cleaned/minimal_pairs_bnc_rand_with_bins.csv",
    target_verbs_info=bnc_verbs_info,
    binned_words=bins_bnc,
    max_lines=140000
)

In [ ]:
create_minimal_pair_with_bin(
    input_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand.wiki.cleaned.csv",
    output_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/WIKI_rand/cleaned/minimal_pairs_wiki_rand_with_bins.csv",
    target_verbs_info=wiki_verbs_info,
    binned_words=bins_wiki,
    max_lines=140000
)

In [ ]:
create_minimal_pair_with_bin(
    input_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/CANDOR_rand/cleaned/minimal_pairs_candor_rand_cleaned.csv",
    output_csv="./evaluation/semantic_minimal_pairs/data/verb_focus/CANDOR_rand/cleaned/minimal_pairs_candor_new_with_bins.csv",
    target_verbs_info=candor_verbs_info,
    binned_words=bins_candor,
    max_lines=140000
)